# 03 Features 
Builds time on ball, first touch, pitch zone, and outcome columns for each reception. 

In [1]:
import pandas as pd
ev = pd.read_pickle("data/nwsl_2023_events.pkl")
ev = ev.sort_values(["match_id", "index"]).reset_index(drop=True)
ev["t"] = pd.to_timedelta(ev["timestamp"]).dt.total_seconds()

reps = pd.read_pickle("data/receptions.pkl")
reps.shape

(112874, 15)

In [2]:
cols = ["index", "t", "type", "player", "possession"]
ev[cols].iloc[100:115]

,index,t,type,player,possession
100,101,75.054,Pass,Emily Ann Fox,2
101,102,76.385,Ball Receipt*,Tyler Lussi,2
102,103,76.385,Carry,Tyler Lussi,2
103,104,76.795,Pass,Tyler Lussi,2
104,105,77.194,Pressure,Lo’eau LaBonta,2
105,106,77.249,Ball Receipt*,Denise O'Sullivan,2
106,107,77.249,Pass,Denise O'Sullivan,2
107,108,78.456,Ball Receipt*,Tyler Lussi,2
108,109,78.456,Carry,Tyler Lussi,2
109,110,80.953,Pass,Tyler Lussi,2


In [3]:
release_types = ["Pass", "Shot", "Dispossessed", "Miscontrol", "Clearance"]

rel = ev[ev["type"].isin(release_types)]
rel = rel[["match_id", "index", "t", "player", "possession", "type"]]
rel.shape

(141261, 6)

In [4]:
r = reps[["match_id", "index", "t", "player", "possession"]].assign(type="Ball Receipt*")

seq = pd.concat([r,rel])
seq = seq.sort_values(["match_id", "index"]).reset_index(drop=True)
seq.head(10)

,match_id,index,t,player,possession,type
0,3881483,5,1.693,Kerolin Nicoli Israel Ferraz,2,Pass
1,3881483,6,3.422,Meredith Speck,2,Ball Receipt*
2,3881483,8,4.190,Meredith Speck,2,Pass
3,3881483,9,5.450,Kiara Shaelene Pickett,2,Ball Receipt*
4,3881483,11,6.261,Kiara Shaelene Pickett,2,Pass
5,3881483,12,8.525,Ryan Williams,2,Ball Receipt*
6,3881483,14,9.390,Ryan Williams,2,Pass
7,3881483,15,11.125,Kaleigh Kurtz,2,Ball Receipt*
8,3881483,17,11.903,Kaleigh Kurtz,2,Pass
9,3881483,18,13.114,Denise O'Sullivan,2,Ball Receipt*


In [6]:
nxt = seq.groupby("match_id").shift(-1)
nxt.head(5)

,index,t,player,possession,type
0,6.0,3.422,Meredith Speck,2.0,Ball Receipt*
1,8.0,4.190,Meredith Speck,2.0,Pass
2,9.0,5.450,Kiara Shaelene Pickett,2.0,Ball Receipt*
3,11.0,6.261,Kiara Shaelene Pickett,2.0,Pass
4,12.0,8.525,Ryan Williams,2.0,Ball Receipt*


In [7]:
same = (
    (nxt["player"] ==seq["player"]) &
    (nxt["possession"] == seq["possession"]) &
    (nxt["type"] != "Ball Receipt*")
)
same.head(5)

0    False
1     True
2    False
3     True
4    False
dtype: bool

In [11]:
seq["time_on_ball"] = (nxt["t"] - seq["t"]).where(same) 
seq["release_type"] = nxt["type"].where(same)
seq.head(5)

,match_id,index,t,player,possession,type,time_on_ball,release_type
0,3881483,5,1.693,Kerolin Nicoli Israel Ferraz,2,Pass,NaN,NaN
1,3881483,6,3.422,Meredith Speck,2,Ball Receipt*,0.768,Pass
2,3881483,8,4.190,Meredith Speck,2,Pass,NaN,NaN
3,3881483,9,5.450,Kiara Shaelene Pickett,2,Ball Receipt*,0.811,Pass
4,3881483,11,6.261,Kiara Shaelene Pickett,2,Pass,NaN,NaN


In [12]:
tob = seq[seq["type"] == "Ball Receipt*"] [["match_id", "index", "time_on_ball", "release_type"]]

reps = reps.merge(tob, on=["match_id", "index"], how ="left")
reps.shape

(112874, 17)